In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gdown
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

DATA_URL = 'https://storage.yandexcloud.net/aiueducation/Content/base/l11/traff.csv'
DATA_FILE = 'traff.csv'

def read_traffic_data(url=DATA_URL, file_path=DATA_FILE):
    gdown.download(url, file_path, quiet=True)
    table = pd.read_csv(
        file_path,
        header=None,
        names=['date', 'value'],
        sep=',',
        thousands=','
    )
    return table.dropna().reset_index(drop=True)

def scale_traffic_values(table):
    source_values = table['value'].to_numpy().reshape(-1, 1)
    value_scaler = MinMaxScaler()
    scaled_values = value_scaler.fit_transform(source_values)
    return source_values, scaled_values, value_scaler

traffic_df = read_traffic_data()
data, data_scaled, scaler = scale_traffic_values(traffic_df)

win_len = 60
batch_size = 32
train_len = int(len(data_scaled) * 0.9)

def make_time_windows(series, length, first_target, last_target):
    x_items = []
    y_items = []

    for target_index in range(first_target, last_target + 1):
        x_items.append(series[target_index - length:target_index])
        y_items.append(series[target_index])

    x_array = np.asarray(x_items, dtype=np.float32)
    y_array = np.asarray(y_items, dtype=np.float32)
    return x_array, y_array

x_train, y_train = make_time_windows(
    data_scaled,
    win_len,
    win_len,
    train_len
)

x_test, y_test = make_time_windows(
    data_scaled,
    win_len,
    train_len + win_len,
    len(data_scaled) - 1
)

train_ds = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)

test_ds = TensorDataset(
    torch.tensor(x_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class TrafficLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm_1 = nn.LSTM(
            input_size=1,
            hidden_size=128,
            batch_first=True
        )
        self.dropout_1 = nn.Dropout(0.2)
        self.lstm_2 = nn.LSTM(
            input_size=128,
            hidden_size=64,
            batch_first=True
        )
        self.dropout_2 = nn.Dropout(0.2)
        self.fc_1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc_2 = nn.Linear(32, 1)

    def forward(self, x):
        x, _ = self.lstm_1(x)
        x = self.dropout_1(x)
        x, _ = self.lstm_2(x)
        x = x[:, -1, :]
        x = self.dropout_2(x)
        x = self.fc_1(x)
        x = self.relu(x)
        x = self.fc_2(x)
        return x

model = TrafficLSTM().to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {
    'loss': [],
    'val_loss': []
}

epochs = 30

for epoch in range(epochs):
    model.train()
    train_losses = []

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        y_pred_batch = model(x_batch)
        loss = loss_fn(y_pred_batch, y_batch)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            y_pred_batch = model(x_batch)
            val_loss = loss_fn(y_pred_batch, y_batch)

            val_losses.append(val_loss.item())

    train_loss = float(np.mean(train_losses))
    valid_loss = float(np.mean(val_losses))

    history['loss'].append(train_loss)
    history['val_loss'].append(valid_loss)

    print(
        f"Эпоха {epoch + 1:02d}/{epochs} | "
        f"loss: {train_loss:.6f} | "
        f"val_loss: {valid_loss:.6f}"
    )

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(history['loss'], label='Обучение')
plt.plot(history['val_loss'], label='Проверка')
plt.title('Ошибка модели')
plt.xlabel('Эпоха')
plt.ylabel('MSE')
plt.legend()
plt.grid()
plt.show()

In [ ]:
def restore_scale(values, value_scaler):
    return value_scaler.inverse_transform(values)

def calc_autocorrelation(series):
    row = np.asarray(series).reshape(-1)
    corr = np.correlate(row, row, mode='full')
    return corr[corr.size // 2:]

def draw_autocorrelation_comparison(real_values, predicted_values, points=100):
    plt.figure(figsize=(15, 6))

    plt.subplot(1, 2, 1)
    plt.plot(calc_autocorrelation(real_values)[:points], label='Оригинал')
    plt.title('Автокорреляция исходного сигнала')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(calc_autocorrelation(predicted_values)[:points], label='Прогноз')
    plt.title('Автокорреляция прогноза')
    plt.legend()

    plt.tight_layout()
    plt.show()

def draw_forecast_part(real_values, predicted_values, points=200):
    plt.figure(figsize=(15, 5))
    plt.plot(real_values[:points], label='Факт')
    plt.plot(predicted_values[:points], label='Предсказание')
    plt.title('Сравнение фрагмента трафика')
    plt.legend()
    plt.grid()
    plt.show()

model.eval()
predicted_batches = []

with torch.no_grad():
    for x_batch, _ in test_loader:
        x_batch = x_batch.to(device)
        predicted_batches.append(model(x_batch).cpu().numpy())

predictions_scaled = np.vstack(predicted_batches)
y_pred = restore_scale(predictions_scaled, scaler)
y_true = data[train_len + win_len:]

draw_autocorrelation_comparison(y_true, y_pred)
draw_forecast_part(y_true, y_pred)

In [ ]:
has_nan_values = np.isnan(data_scaled).any()
print("Есть ли пустые значения в данных:", has_nan_values)